In [1]:
!pip -q install datasets pandas

In [2]:
from datasets import load_dataset

base = "https://huggingface.co/datasets/PolyAI/banking77/resolve/main/data"
banking = load_dataset(
    "parquet",
    data_files={
        "train":    f"{base}/train-00000-of-00001.parquet",
        "test":     f"{base}/test-00000-of-00001.parquet",
        },
)


print(banking)
print(banking['train'][0])
print(banking['test'][0])

labels_names = banking['train'].features['label'].names
print("no_of_intents:", len(labels_names))
print("first_examples_intent:", labels_names[banking["train"][0]["label"]])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})
{'text': 'I am still waiting on my card?', 'label': 11}
{'text': 'How do I locate my card?', 'label': 11}
no_of_intents: 77
first_examples_intent: card_arrival


In [3]:
from collections import Counter

label_names = banking["train"].features["label"].names

for split in ["train", "test"]:
    counts = Counter(banking[split]["label"])
    empty_messages = sum(
        not text.strip() for text in banking[split]["text"]
    )

    print(f"\n{split.upper()}")
    print("Messages:", len(banking[split]))
    print("Intents represented:", len(counts))
    print("Fewest examples in an intent:", min(counts.values()))
    print("Most examples in an intent:", max(counts.values()))
    print("Empty messages:", empty_messages)

print("\nFive example messages:")
for index in [0, 100, 500, 1000, 2000]:
    row = banking["train"][index]
    print(f"{index}: [{label_names[row['label']]}] {row['text']}")


TRAIN
Messages: 10003
Intents represented: 77
Fewest examples in an intent: 35
Most examples in an intent: 187
Empty messages: 0

TEST
Messages: 3080
Intents represented: 77
Fewest examples in an intent: 40
Most examples in an intent: 40
Empty messages: 0

Five example messages:
0: [card_arrival] I am still waiting on my card?
100: [card_arrival] can you share card tracking number?
500: [card_payment_wrong_exchange_rate] Why is the exchange rate different from what I expected?
1000: [fiat_currency_support] What are the currency types that I can keep money in.
2000: [pending_top_up] My top is still pending.


In [4]:
from collections import defaultdict
import random

SEED = 42

def balanced_indices(split, examples_per_intent, seed):
    by_intent = defaultdict(list)

    for index, label in enumerate(split["label"]):
        by_intent[label].append(index)

    rng = random.Random(seed)
    selected = []
    for label in range(len(by_intent)):
        selected.extend(
        rng.sample(by_intent[label], examples_per_intent)
        )
    return sorted(selected)

dev_indices = balanced_indices(banking["train"], 1, SEED)
final_indices = balanced_indices(banking["train"], 2, SEED)


dev_set = banking["train"].select(dev_indices)
final_set = banking["train"].select(final_indices)

assert len(dev_set) == 77
assert len(final_set) == 154
assert len(dev_set["label"]) == 77
assert len(final_set["label"]) == 154

print("seed:", SEED)
print("dev set length:", len(dev_set))
print("final set length:", len(final_set))
print(" first 5 dev indices:", dev_indices[:5])
print(" first 5 final indices:", final_indices[:5])


seed: 42
dev set length: 77
final set length: 154
 first 5 dev indices: [108, 160, 381, 533, 658]
 first 5 final indices: [107, 139, 154, 224, 321]


In [5]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [6]:
!pip -q install -U transformers accelerate

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype = torch.float16,
    device_map = "auto",
)

print("loaded:", model_name)
print("model_device:", model.device)

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

loaded: Qwen/Qwen2.5-1.5B-Instruct
model_device: cuda:0


 the tokenizer converts text into token IDs; the pretrained causal model generates the next tokens; float16 reduces GPU memory use; and device_map="auto" places the model on the available device. The first run downloads model files, so it may take a few minutes.

In [15]:
def baseline_reply(message):
    prompt = f"Identify the banking intent and respond to the customer:\n{message}"

    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,  # Greedy decoding: reproducible baseline
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


for index in [0, 25, 50]:
    row = dev_set[index]
    print(f"\n--- Development example {index} ---")
    print("Customer:", row["text"])
    print("Expected intent:", label_names[row["label"]])
    print("Baseline output:", baseline_reply(row["text"]))



--- Development example 0 ---
Customer: Is it possible for me to track the card is was mailed to me?
Expected intent: card_arrival
Baseline output: Yes, you can usually track the mailing of your bank card through your account or contact information provided when you received the card. If you have access to this information, you should be able to check its delivery status online. Alternatively, if you need further assistance with tracking, please provide more details about how you received the card (e.g., physical mail, email) so I can guide you accordingly.

--- Development example 25 ---
Customer: What is the status of my top up?
Expected intent: top_up_reverted
Baseline output: I'm sorry, but I cannot provide an accurate response without more information about your specific situation. Could you please provide me with some additional details such as the name of your bank or the type of account you have? This will help me better understand your request and provide you with a more appr

These outputs give us three useful baseline findings:

**Example 0**: Helpful general guidance, but no exact card_arrival label. It also suggests account-based tracking without knowing whether that feature exists.

**Example 25**: The model correctly avoids claiming it knows the top-up status. However, the message alone does not clearly establish top_up_reverted; that label may be impossible to infer reliably from this wording. Keep this as an ambiguity case, not simply a model mistake.

**Example 50**: It describes an intent instead of naming verify_source_of_funds, then starts listing unsupported possibilities. The output also appears cut off at the 120-token limit.

Next, test one zero-shot prompt. **“Zero-shot”** means giving instructions and allowed labels but no worked examples. Keep the same three development messages and greedy decoding:

In [16]:
def zero_shot_reply(message):
    labels = ", ".join(label_names)
    prompt = (
        "Choose the single best BANKING77 intent for the customer message "
        "from the allowed labels below. Then write a brief, general reply. "
        "Do not claim to know the customer's account status.\n\n"
        f"Allowed labels: {labels}\n\n"
        f"Customer message: {message}\n\n"
        "Use this format:\nIntent: <exact allowed label>\nReply: <brief reply>"
    )

    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=160,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(
        output[0, inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )


for index in [0, 25, 50]:
    row = dev_set[index]
    print(f"\n--- Development example {index} ---")
    print("Customer:", row["text"])
    print("Expected intent:", label_names[row["label"]])
    print("Zero-shot output:", zero_shot_reply(row["text"]))


--- Development example 0 ---
Customer: Is it possible for me to track the card is was mailed to me?
Expected intent: card_arrival
Zero-shot output: Intent: get_physical_card
Reply: We can help you track your physical card that has been mailed to you. Please provide us with your tracking number or any other relevant information so we can assist further.

--- Development example 25 ---
Customer: What is the status of my top up?
Expected intent: top_up_reverted
Zero-shot output: Intent: top_up_status
Reply: We can help you check your top-up status. Please provide us with your account number or any other relevant information so we can assist you further.

--- Development example 50 ---
Customer: Where do my funds originate from?
Expected intent: verify_source_of_funds
Zero-shot output: Intent: verify_source_of_funds
Reply: To help you understand where your funds come from, we need to verify the source of your funds. Please provide us with any additional information or documents that may 